In [16]:
#Step1
import pandas as pd
import pandasql as ps
import pixiedust
import sys
sys.path.append('..')
import util
import etl
import pyarrow.parquet as pq
import pyarrow as pa
!pip install duckdb
import duckdb
import inspect

# Get the path of the imported module (etl.py)
etl_module_path = inspect.getfile(etl)
print("Path of the imported ETL module (etl.py):", etl_module_path)
util.usedatabase(spark, "real_world_data_jun_2022")

pixiedust.enableJobMonitor()

con = duckdb.connect()

▸,:,


Please see https://github.com/pypa/pip/issues/5599 for advice on fixing the underlying issue.
To avoid this problem you can invoke Python with '-m pip' instead of running pip directly.
Path of the imported ETL module (etl.py): /home/o_suchsi/work/Oklahoma State/Priya/epilepsy/etl.py
Using real_world_data_jun_2022 ....
Spark Job Progress Monitor already enabled


In [17]:
#Reading-> Commo-Cohort
Como_Result_Final2 = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Cohort_Paired_Commo_RemComma")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
Como_Result_Final2.createOrReplaceTempView('Epilepsy_Cohort_Como')

▸,:,


In [19]:
#Reading-> Commo-Control
Como_Result_Final1 = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Control_Paired_Commo_RemComma")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
Como_Result_Final1.createOrReplaceTempView('Epilepsy_Control_Como')

▸,:,


In [21]:
##############################################Final dictionary to be used for control commo conversion process #########################
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

# Read the CSV file into a DataFrame
df = spark.read.csv("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/ICDconvertion.csv", header=True, inferSchema=True)

# Define a UDF to cast the "icd9cm" column to StringType
def cast_to_string(icd9cm):
    return str(icd9cm)

# Register the UDF
cast_to_string_udf = F.udf(cast_to_string, StringType())

# Apply the UDF to cast "icd9cm" to StringType
df = df.withColumn("icd9cm", cast_to_string_udf(df["icd9cm"]))

# Convert DataFrame to a Pandas DataFrame
pandas_df = df.toPandas()

# Convert Pandas DataFrame to a dictionary
icd_dict = pandas_df.set_index("icd9cm")["icd10cm"].to_dict()

# Show the resulting dictionary
print(icd_dict)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

{'10': 'A000', '11': 'A001', '19': 'A009', '20': 'A0100', '21': 'A011', '22': 'A012', '23': 'A013', '29': 'A014', '30': 'A020', '31': 'A021', '320': 'A360', '321': 'A361', '322': 'A3689', '323': 'A362', '324': 'A0224', '329': 'A369', '38': 'A028', '39': 'A029', '40': 'A030', '41': 'A031', '42': 'B20', '43': 'A033', '48': 'A880', '49': 'A039', '50': 'A050', '51': 'A051', '52': 'A052', '53': 'A058', '54': 'A053', '581': 'A055', '589': 'A058', '59': 'A059', '60': 'A060', '61': 'A90', '62': 'A062', '63': 'A064', '64': 'A852', '65': 'A066', '66': 'A067', '68': 'A0689', '69': 'A069', '70': 'A070', '71': 'A829', '72': 'A073', '73': 'A078', '74': 'A072', '75': 'B2790', '78': 'A078', '79': 'A079', '800': 'A044', '801': 'A040', '802': 'A041', '803': 'A042', '804': 'A043', '809': 'A044', '81': 'A048', '82': 'A048', '83': 'A048', '841': 'B519', '842': 'B529', '843': 'B530', '844': 'B538', '845': 'B529', '846': 'B54', '847': 'B538', '849': 'B528', '85': 'A049', '861': 'B575', '862': 'B571', '863': 

<IPython.core.display.Javascript object>

In [22]:
Como_Result_Test = spark.sql(""" 
    SELECT
        DISTINCT
        comorbidityid   
    FROM
        Epilepsy_Cohort_Como
""")
Como_Result_Test.show()
# Collect the distinct comorbidityid values into a list
comorbidityid_list1 = Como_Result_Test.rdd.flatMap(lambda x: x).collect()

# Show the distinct comorbidityid values
print(comorbidityid_list1)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+-------------+
|comorbidityid|
+-------------+
|       M23.92|
|      Z87.820|
|     S83.231A|
|       E83.42|
|        850.9|
|       787.20|
|       Z76.89|
|        R51.0|
|        V14.2|
|       H55.81|
|       M62.89|
|       Z51.11|
|       M06.09|
|        V17.1|
|        Z59.6|
|       K85.90|
|          Z21|
|       M50.31|
|        N81.4|
|       997.62|
+-------------+
only showing top 20 rows



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

['M23.92', 'Z87.820', 'S83.231A', 'E83.42', '850.9', '787.20', 'Z76.89', 'R51.0', 'V14.2', 'H55.81', 'M62.89', 'Z51.11', 'M06.09', 'V17.1', 'Z59.6', 'K85.90', 'Z21', 'M50.31', 'N81.4', '997.62', 'S60.021A', '959.4', 'M67.441', '521.81', 'M43.10', 'K66.8', '286.9', 'N35.911', 'S59.919A', 'M86.8X0', 'S02.610A', 'F12.929', '803.00', '840.8', 'M31.6', 'N99.89', 'H35.32', 'S63.592A', 'C84.60', '783.43', '536.8', '810.03', 'G03.9', '458.29', 'Q27.30', 'W13.3XXD', 'E11.359', 'Z45.89', 'M25.011', 'Y90.4', '958.3', 'N60.31', 'B17.9', '733.01', 'F15.11', '110.9', 'S82.002D', '288142002', 'E006.9', 'M77.40', '781.9', 'O36.8990', '66857006', 'Q79.9', 'S52.209D', '191.9', 'L89.302', 'V57.6XXA', 'S63.681A', 'J20.6', 'S02.632A', 'S06.4X2A', 'C16.0', 'N64.0', 'I72.1', '886.0', '723.3', 'M00.272', 'M60.819', 'I26.93', 'H02.009', 'H60.559', '438.10', '133933007', '25510005', '471.9', 'G52.9', 'K08.419', 'E80.20', 'H52.02', '86406008', 'M24.011', 'S42.201S', 'O92.29', 'M51.87', '25003006', 'S89.121D', '4

<IPython.core.display.Javascript object>

In [23]:
Como_Result_Test = spark.sql(""" 
    SELECT
        DISTINCT
        comorbidityid   
    FROM
        Epilepsy_Control_Como
""")
Como_Result_Test.show()
# Collect the distinct comorbidityid values into a list
comorbidityid_list = Como_Result_Test.rdd.flatMap(lambda x: x).collect()

# Show the distinct comorbidityid values
print(comorbidityid_list)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

['Z76.89', '850.9', 'Z87.820', '783.43', 'F12.929', 'E006.9', 'Z59.6', 'E83.42', '810.03', 'M43.10', 'H55.81', 'Z45.89', '458.29', 'M62.89', '959.4', 'V14.2', 'K85.90', 'S60.021A', '840.8', 'M50.31', 'Z51.11', 'Y90.4', 'S00.252A', '536.8', '803.00', 'N81.4', '27503000', 'S83.231A', '787.20', '733.01', 'B17.9', 'Z21', 'S59.919A', 'F15.11', 'Q26.3', '416113008', 'Q27.30', 'J20.6', 'N60.31', 'M99.52', '110.9', '133933007', 'S61.012D', 'H35.32', 'I87.012', '720016003', 'S02.632A', 'S63.592A', 'G03.9', 'K08.419', 'M23.92', 'M51.87', '425988004', '21850008', 'V17.1', 'N99.89', 'M31.6', '928.11', 'T50.B94A', '95607001', '238150007', 'S81.032A', 'V54.10', '286.9', 'O92.29', 'S63.681A', 'C17.9', '958.3', 'M77.40', '66176005', 'I26.93', '471.9', 'S73.001A', '432788009', '691', 'Q53.20', '781.9', 'S82.002D', '521.81', 'M67.441', 'G52.9', '788.65', 'I72.1', 'S62.350A', '288.02', 'F68.11', '807.05', '415.11', 'H10.819', 'M06.09', '886.0', 'N35.911', '621.3', 'Q79.9', 'L89.302', 'O36.8990', 'R51.0',

<IPython.core.display.Javascript object>

In [24]:
# Combine the lists and extract distinct values
combined_and_distinct = list(set(comorbidityid_list + comorbidityid_list1))

# Initialize a counter for items containing commas
comma_count = 0

# Iterate through the list and print values containing commas
for item in combined_and_distinct:
    if ',' in item:
        print(item)
        comma_count += 1

# Print the total number of items in the combined_and_distinct list
print("Total number of distinct items:", len(combined_and_distinct))
print("Number of items containing commas:", comma_count)

▸,:,


Total number of distinct items: 79274
Number of items containing commas: 0


In [25]:
##################################Final Package to be used to normalize commorbidity for cohort group #########################
from pyspark.sql.functions import col, udf, split, regexp_replace, explode
from pyspark.sql.types import StringType

# Define a Python function to process comorbidityid
def process_comorbidityid(comorbidityid):
    # Check if comorbidityid contains a comma
    if ',' in comorbidityid:
        # Split comorbidityid by commas
        comorbidity_list = comorbidityid.split(',')
        # Process individual values and map them to the dictionary
        processed_comorbidities = [process_individual_comorbidity(cid) for cid in comorbidity_list]
        # Return the processed values as a comma-separated string
        return ','.join(processed_comorbidities)
    
    # Check if comorbidityid contains a decimal point
    if '.' in comorbidityid:
        # Remove the decimal point
        comorbidityid = comorbidityid.replace('.', '')

    # Look up comorbidityid in the dictionary and return the mapped value if found
    mapped_value = icd_dict.get(comorbidityid, None)
    if mapped_value is not None:
        return mapped_value  # Remove square brackets and return as is

    # If no match found, retain the original comorbidityid
    return comorbidityid

# Define a function to process individual comorbidity values
def process_individual_comorbidity(comorbidity):
    # Remove decimal points
    comorbidity = comorbidity.replace('.', '')
    # Strip square brackets and leading/trailing spaces
    comorbidity = comorbidity.strip('[]').strip()
    # Look up comorbidityid in the dictionary and return the mapped value if found
    mapped_value = icd_dict.get(comorbidity, None)
    if mapped_value is not None:
        return mapped_value
    # If no match found, retain the original comorbidity
    return comorbidity

# Register the process_comorbidityid function as a UDF
process_comorbidityid_udf = udf(process_comorbidityid, StringType())

# Assuming you have a DataFrame named Como_Result_Final1
# Create a new column to store the original comorbidityid values
result_df_full = Como_Result_Final2.withColumn("original_comorbidityid", col("comorbidityid"))

# Apply the UDF to the comorbidityid column and create a new column
result_df_full = result_df_full.withColumn("comorbidityid", process_comorbidityid_udf(col("comorbidityid")))

# Remove square brackets from comorbidityid values
result_df_full = result_df_full.withColumn("comorbidityid", regexp_replace(col("comorbidityid"), r'\[|\]', ''))

# Split the comorbidityid values by comma and explode the resulting array
result_df_full = result_df_full.withColumn("comorbidityid", split(col("comorbidityid"), ",")) \
              .withColumn("comorbidityid", explode(col("comorbidityid")))

# Show the resulting DataFrame
result_df_full.show(100,truncate=False)

▸,:,


<IPython.core.display.Javascript object>

+------------------------------------+-------------+----------------------+
|personid                            |comorbidityid|original_comorbidityid|
+------------------------------------+-------------+----------------------+
|a3d8d352-e70e-4fc7-bb2d-147b490a6e2e|S32433A      |S32.433A              |
|3ebf18f1-5824-4940-9fd3-0f0db8a7e19f|S0183XA      |S01.83XA              |
|0e8af23f-0507-4385-9c2b-ef112cc3942d|M25562       |M25.562               |
|f3ddb599-de0d-46a8-8a89-8f563ff5bbcd|I10          |I10                   |
|643ed68b-e008-42df-9a19-2d1efc9bb1ef|M25561       |M25.561               |
|639fd53f-8388-4abf-a4e8-1891ba4ac9d7|D631         |D63.1                 |
|7b774ef0-f3f0-4dce-bc54-fbb6b32a7528|S5292XE      |S52.92XE              |
|87656633-381e-4126-83a2-af4363b4fe1b|M25471       |M25.471               |
|3b88b4b8-a6a1-4d7c-8925-71c507f197d9|E785         |E78.5                 |
|88001fb3-9bac-42c0-b8b9-cd77997ea7fa|S36116A      |S36.116A              |
|04fbb148-a9

<IPython.core.display.Javascript object>

In [26]:
##################################Final Control Commo Replaced Codes ############################################################
result_df_full.write.mode("overwrite").parquet("Normalized_Cohort_Commo_AFterCommaRemoval")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [27]:
!hadoop fs -copyToLocal Normalized_Cohort_Commo_AFterCommaRemoval ~/work/Oklahoma%20State/Priya/epilepsy

▸,:,


In [10]:
##################################Final Package to be used to normalize commorbidity for control group #########################
from pyspark.sql.functions import col, udf, split, regexp_replace, explode
from pyspark.sql.types import StringType
from pyspark.sql import SparkSession

# Define a Python function to process comorbidityid
def process_comorbidityid(comorbidityid):
    # Check if comorbidityid contains a comma
    if ',' in comorbidityid:
        # Split comorbidityid by commas
        comorbidity_list = comorbidityid.split(',')
        # Process individual values and map them to the dictionary
        processed_comorbidities = [process_individual_comorbidity(cid) for cid in comorbidity_list]
        # Return the processed values as a comma-separated string
        return ','.join(processed_comorbidities)
    
    # Check if comorbidityid contains a decimal point
    if '.' in comorbidityid:
        # Remove the decimal point
        comorbidityid = comorbidityid.replace('.', '')

    # Look up comorbidityid in the dictionary and return the mapped value if found
    mapped_value = icd_dict.get(comorbidityid, None)
    if mapped_value is not None:
        return mapped_value  # Remove square brackets and return as is

    # If no match found, retain the original comorbidityid
    return comorbidityid

# Define a function to process individual comorbidity values
def process_individual_comorbidity(comorbidity):
    # Remove decimal points
    comorbidity = comorbidity.replace('.', '')
    # Strip square brackets and leading/trailing spaces
    comorbidity = comorbidity.strip('[]').strip()
    # Look up comorbidityid in the dictionary and return the mapped value if found
    mapped_value = icd_dict.get(comorbidity, None)
    if mapped_value is not None:
        return mapped_value
    # If no match found, retain the original comorbidity
    return comorbidity

# Register the process_comorbidityid function as a UDF
process_comorbidityid_udf = udf(process_comorbidityid, StringType())

# Assuming you have a DataFrame named Como_Result_Final1
# Create a new column to store the original comorbidityid values
result_df_full = Como_Result_Final1.withColumn("original_comorbidityid", col("comorbidityid"))

# Apply the UDF to the comorbidityid column and create a new column
result_df_full = result_df_full.withColumn("comorbidityid", process_comorbidityid_udf(col("comorbidityid")))

# Remove square brackets from comorbidityid values
result_df_full = result_df_full.withColumn("comorbidityid", regexp_replace(col("comorbidityid"), r'\[|\]', ''))

# Split the comorbidityid values by comma and explode the resulting array
result_df_full = result_df_full.withColumn("comorbidityid", split(col("comorbidityid"), ",")) \
              .withColumn("comorbidityid", explode(col("comorbidityid")))

# Show the resulting DataFrame
result_df_full.show(100,truncate=False)

▸,:,


<IPython.core.display.Javascript object>

+------------------------------------+-------------+----------------------+
|personid                            |comorbidityid|original_comorbidityid|
+------------------------------------+-------------+----------------------+
|38405b9f-7e14-410c-be77-5217dcc8cf15|F99          |F99                   |
|f65c7d2b-3e52-4e36-9acd-3720b1dc6b2b|R519         |R51.9                 |
|5d5ebbb0-aadc-43c2-99a2-755124f287b1|Z6838        |Z68.38                |
|4fbbb6b9-120a-4e78-a859-8b2eec6d41f8|Z992         |Z99.2                 |
|3c279f74-23bd-490b-8e7b-4ccdcb98c071|H52203       |H52.203               |
|304bc3ba-6d93-4576-b25e-a01135c7ba4d|M25522       |M25.522               |
|08b5b1cd-115c-429b-99c7-fab054ab91aa|B9689        |B96.89                |
|982e1c47-bf15-4e54-b9ab-19f39c265fc7|G8929        |G89.29                |
|a644d757-e134-4dea-9127-73ba1b438c43|T782XXS      |T78.2XXS              |
|ced3f036-4ada-4bf5-b0c9-ff3c57b4f133|M79602       |M79.602               |
|7228e86f-2c

<IPython.core.display.Javascript object>

In [11]:
##################################Final Control Commo Replaced Codes ############################################################
result_df_full.write.mode("overwrite").parquet("Normalized_Control_Commo_AFterCommaRemoval")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
!hadoop fs -copyToLocal Normalized_Control_Commo_AFterCommaRemoval ~/work/Oklahoma%20State/Priya/epilepsy

▸,:,


copyToLocal: `/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Normalized_Control_Commo_AFterCommaRemoval/_SUCCESS': File exists
copyToLocal: `/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Normalized_Control_Commo_AFterCommaRemoval/part-00000-f17d0c0d-9995-4cfc-8be5-0c89bc359361-c000.snappy.parquet': File exists
copyToLocal: `/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Normalized_Control_Commo_AFterCommaRemoval/part-00001-f17d0c0d-9995-4cfc-8be5-0c89bc359361-c000.snappy.parquet': File exists
copyToLocal: `/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Normalized_Control_Commo_AFterCommaRemoval/part-00002-f17d0c0d-9995-4cfc-8be5-0c89bc359361-c000.snappy.parquet': File exists
copyToLocal: `/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Normalized_Control_Commo_AFterCommaRemoval/part-00003-f17d0c0d-9995-4cfc-8be5-0c89bc359361-c000.snappy.parquet': File exists
copyToLocal: `/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Normalized_Control_Commo_AFterCommaRemoval/part-00

In [40]:
# Assuming you have a DataFrame named "df" with the column you want to rename
combined_Final = combined_Final.withColumnRenamed("comorbidityid", "comorbidityid_RF")
combined_Final = combined_Final.withColumnRenamed("original_comorbidityid", "comorbidityid")
combined_Final.printSchema()

▸,:,


root
 |-- personid: string (nullable = true)
 |-- comorbidityid_RF: string (nullable = true)
 |-- comorbidityid: string (nullable = true)



In [15]:
!hadoop fs -copyToLocal Normalized_Cohort_Commo_AFterCommaRemoval.parquet ~/work/Oklahoma%20State/Priya/epilepsy

▸,:,


copyToLocal: `Normalized_Cohort_Commo_AFterCommaRemoval.parquet': No such file or directory
